YOLO 26n (100 EPOCHS, 1280 IMGSZ)

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Load YOLO26n model
model = YOLO("yolo26n.pt")

In [ ]:
# Train the model on your custom dataset from Google Drive
results = model.train(data="/content/drive/MyDrive/DATASET 3 - Pork Rasher/data.yaml", epochs=100, imgsz=1280)

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

# Path to your YOLO training results
results_dir = "/content/runs/detect/train"

# Evaluation plots to display
plot_files = [
    "results.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxPR_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png"
]

# Create a figure with 2 columns
fig, axes = plt.subplots(4, 2, figsize=(18, 24))
axes = axes.flatten()

for i, filename in enumerate(plot_files):
    filepath = os.path.join(results_dir, filename)

    if os.path.exists(filepath):
        img = Image.open(filepath)
        axes[i].imshow(img)
        axes[i].set_title(
            filename.replace("_", " ").replace(".png", ""),
            fontsize=14,
            fontweight="bold"
        )
    else:
        axes[i].text(
            0.5, 0.5,
            f"{filename}\nNot found",
            ha="center",
            va="center"
        )

    axes[i].axis("off")

# Hide unused subplot
for j in range(len(plot_files), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/best.pt")

network = model.model
network.eval()

device = next(network.parameters()).device

# Use a dummy image of size 640x640
dummy_input = torch.zeros(
    1, 3, 640, 640,
    device=device
)

layer_info = []
hooks = []

def make_hook(name):
    def hook(module, inputs, output):
        if isinstance(output, torch.Tensor):
            shape = tuple(output.shape)

            # Number of activations per image
            activations = output[0].numel()

            layer_info.append({
                "Layer": name,
                "Type": type(module).__name__,
                "Output Shape": shape,
                "Activations": activations
            })
    return hook

# Register hooks on leaf modules
for name, module in network.named_modules():
    if name and len(list(module.children())) == 0:
        hooks.append(
            module.register_forward_hook(make_hook(name))
        )

# Perform forward pass
with torch.no_grad():
    _ = network(dummy_input)

# Remove hooks
for hook in hooks:
    hook.remove()

# Display results
import pandas as pd

df = pd.DataFrame(layer_info)

display(df)

print("Total recorded activations:",
      f"{df['Activations'].sum():,}")

In [ ]:
from google.colab import sheets

# Convert 'Output Shape' column to string to prevent APIError with Google Sheets upload
df_sheets = df.copy()
df_sheets['Output Shape'] = df_sheets['Output Shape'].astype(str)

sheet = sheets.InteractiveSheet(df=df_sheets)

In [ ]:
# Calculate model statistics
total_params = sum(p.numel() for p in network.parameters())
tuned_params = sum(p.numel() for p in network.parameters() if p.requires_grad)
total_layers = len(list(network.modules()))

# Count neurons as total output features/channels from layers with weight/bias units
total_neurons = 0
for module in network.modules():
    if hasattr(module, 'out_channels'):
        total_neurons += module.out_channels
    elif hasattr(module, 'out_features'):
        total_neurons += module.out_features

# Total activations from our previous forward pass analysis
total_activations = df['Activations'].sum()

# Display summary
print("=" * 50)
print("               YOLO26n TRAINING SUMMARY            ")
print("=" * 50)
print(f"{'- Total Layers:':<30} {total_layers:,}")
print(f"{'- Total Neurons (Features):':<30} {total_neurons:,}")
print(f"{'- Total Activations (640x640):':<30} {total_activations:,}")
print(f"{'- Total Parameters:':<30} {total_params:,}")
print(f"{'- Tuned (Trainable) Parameters:':<30} {tuned_params:,}")
print("=" * 50)

In [ ]:
import torch

# 1. Retrieve FLOPs
# YOLO26n reports 5.4 GFLOPs at 640x640 input resolution
total_flops = 5.4 * 1e9

# 2. Calculate Memory Footprint (Data Transferred in Bytes)
# Assumes 32-bit float (4 Bytes) precision
bytes_per_param = 4
bytes_per_activation = 4

total_params = 2535830
total_activations = 63685600

param_memory = total_params * bytes_per_param
# Activations are read and written once during execution
activation_memory = total_activations * bytes_per_activation * 2
total_memory_accessed_bytes = param_memory + activation_memory

# 3. Calculate Arithmetic Intensity (FLOPs / Byte)
arithmetic_intensity = total_flops / total_memory_accessed_bytes

# 4. Roofline analysis
# Modern GPUs (like T4, A100, Blackwell) have an operational ridge point between 20 to 150 FLOPs/Byte.
# We will use a standard ridge point of 25.0 FLOPs/Byte for typical accelerator hardware.
ridge_point = 25.0
bound_type = "COMPUTE-BOUNDED" if arithmetic_intensity > ridge_point else "MEMORY-BOUNDED"

# Print Results
print("=" * 60)
print("          ARITHMETIC INTENSITY & ROOFLINE ANALYSIS          ")
print("=" * 60)
print(f"- Total FLOPs:                     {total_flops:,.0f} FLOPs ({total_flops/1e9:.2f} GFLOPs)")
print(f"- Parameter Memory:                {param_memory:,.0f} Bytes ({param_memory/(1024**2):.2f} MB)")
print(f"- Activation Memory (R/W):         {activation_memory:,.0f} Bytes ({activation_memory/(1024**2):.2f} MB)")
print(f"- Total Data Movement:             {total_memory_accessed_bytes:,.0f} Bytes ({total_memory_accessed_bytes/(1024**2):.2f} MB)")
print(f"- Arithmetic Intensity:            {arithmetic_intensity:.4f} FLOPs/Byte")
print(f"- Assumed Hardware Ridge Point:     {ridge_point:.1f} FLOPs/Byte")
print("-" * 60)
print(f"CONCLUSION: The model operations are {bound_type}!")
print("=" * 60)

In [ ]:
import shutil
from google.colab import files

# Zip the folder /content/runs into runs.zip
shutil.make_archive("/content/runs", "zip", "/content/runs")

# Download the zipped file
files.download("/content/runs.zip")